Here we solve the Stokes flow problem without self-gravitation. 

In [1]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy.special import eval_legendre, roots_jacobi

The Earth's mantle is modelled as a shell $c < r < b$. There is a viscosity jump at the MTZ (670 km). There is also a Gaussian density anomaly.

In [2]:
c, b, l = 0.55, 1.0, 2
r670 = 0.8
L = l * (l + 1)
n = 5  # polynomial degree of the velocity (pressure: n - 2)
drmax = 0.02

In [3]:
ne_low = int(np.ceil((r670 - c) / drmax))  # Number of elements in lower mantle.
ne_up = int(np.ceil((b - r670) / drmax))  # Number of elements in upper mantle.
xe = np.concatenate(
    (np.linspace(c, r670, ne_low + 1), np.linspace(r670, b, ne_up + 1)[1:])
)
ne = len(xe) - 1
nq = n + 1


We now define the reference elements on $-1 \leq \xi \leq 1$. 

In [4]:
xi = np.concatenate(([-1.0], np.sort(roots_jacobi(n - 1, 1.0, 1.0)[0]), [1.0]))
wq = 2.0 / (
    n * (n + 1) * eval_legendre(n, xi) ** 2
)  # quadrature weights (nodes = quad points)

The derivative matrix is needed: `D[i, j] = P_n(xi_i) / (P_n(xi_j)(xi_i - xi_j))`

In [5]:
Pn = eval_legendre(n, xi)
D = np.zeros((nq, nq))
for i in range(nq):
    for j in range(nq):
        if i != j:
            D[i][j] = Pn[i] / (Pn[j] * (xi[i] - xi[j]))
for i in range(nq):  # Since every row of D must sum to zero.
    row_sum = 0.0
    for j in range(nq):
        if j != i:
            row_sum += D[i][j]
    D[i][i] = -row_sum

We now consider the pressure, which has two degrees of freedom less than the velocity. 

In [6]:
xp = np.concatenate(([-1.0], np.sort(roots_jacobi(n - 3, 1.0, 1.0)[0]), [1.0]))
npe = len(xp)
Bp = np.ones((n + 1, npe))
for j in range(npe):
    for m in range(npe):
        if m != j:
            Bp[:, j] *= (xi - xp[m]) / (xp[j] - xp[m])

Each column of `Bp` is a pressure basis function evaluated at the quadrature points. 

In [7]:
Nv = ne * n + 1
Np = ne * npe
oU, oV, oP = 0, Nv, 2 * Nv
N = 2 * Nv + Np
nloc_v = 2 * nq  # local velocity dofs: U (nq) then V (nq)
nloc = nloc_v + npe  # plus the local pressure dofs
print(f"{ne} elements, {N} unknowns  (U: {Nv}, V: {Nv}, P: {Np})")


23 elements, 324 unknowns  (U: 116, V: 116, P: 92)


We now build up the local matrix and scatter it into the global matrix. 

In [8]:
A = np.zeros((N, N))  # forward matrix (dense)
bvec = np.zeros(N)  # forward load
elements = []  # everything the adjoint and the kernels need, per element
wk = [
    1.0,
    0.5,
    L / 2.0,
    L * (L - 2.0) / 2.0,
    -1.0 / 3.0,
]  # weights of the five strain terms

for e in range(ne):
    h = xe[e + 1] - xe[e]
    r = [0.0] * nq
    w = [0.0] * nq
    eta = [0.0] * nq
    rho0 = [0.0] * nq
    kappa = [0.0] * nq
    g0 = [0.0] * nq
    f = [0.0] * nq

    for q in range(nq):
        xq = xi[q]
        if q == 0:
            xq += 1e-12  # nudge the end points inside the element
        if q == nq - 1:
            xq -= 1e-12
        r[q] = xe[e] + (xq + 1.0) * h / 2.0
        w[q] = wq[q] * h / 2.0
        if r[q] < r670:
            eta[q] = 30.0
        else:
            eta[q] = 1.0
        rho0[q] = np.exp(0.4 * (1.0 - r[q]))
        kappa[q] = -0.4 * r[q]
        g0[q] = 1.0
        f[q] = -g0[q] * np.exp(-(((r[q] - 0.75) / 0.05) ** 2))

    Q = [[[0.0] * nloc_v for _ in range(nq)] for _ in range(5)]
    Qanel = [[0.0] * nloc_v for _ in range(nq)]  # div u + kappa U / r
    for q in range(nq):
        for j in range(nq):
            dd = (2.0 / h) * D[q][
                j
            ]  # d/dr of the Lagrange basis function j at point q
            delta = (
                1.0 if q == j else 0.0
            )  # basis function j is 1 at its own node, 0 elsewhere
            Q[0][q][j] = dd  # U'
            Q[1][q][j] = 2.0 / r[q] * delta  # F, U part
            Q[1][q][nq + j] = -L / r[q] * delta  # F, V part
            Q[2][q][j] = 1.0 / r[q] * delta  # X, U part
            Q[2][q][nq + j] = dd - 1.0 / r[q] * delta  # X, V part
            Q[3][q][nq + j] = 1.0 / r[q] * delta  # V / r
        for a in range(nloc_v):
            Q[4][q][a] = Q[0][q][a] + Q[1][q][a]  # div u
            Qanel[q][a] = Q[4][q][a]
        for j in range(nq):
            if q == j:
                Qanel[q][j] += kappa[q] / r[q]  # + kappa U / r
    elements.append(
        {
            "r": r,
            "w": w,
            "eta": eta,
            "rho0": rho0,
            "g0": g0,
            "f": f,
            "Q": Q,
            "Qanel": Qanel,
        }
    )


In [ ]:
for e in range(ne):
    r, w, eta, rho0, f = (
        elements[e]["r"],
        elements[e]["w"],
        elements[e]["eta"],
        elements[e]["rho0"],
        elements[e]["f"],
    )
    Q, Qanel = elements[e]["Q"], elements[e]["Qanel"]
    K = np.zeros((nloc, nloc))
    # This is the viscous block.
    for k in range(5):
        for q in range(nq):
            factor = wk[k] * 2.0 * eta[q] * r[q] ** 2 * w[q]
            for a in range(nloc_v):
                for bb in range(nloc_v):
                    K[a][bb] += Q[k][q][a] * factor * Q[k][q][bb]
    # This is the pressure block.
    for q in range(nq):
        for a in range(nloc_v):
            for p in range(npe):
                # -int r^2 P div u~ : rows test velocity a, column pressure p
                K[a][nloc_v + p] -= Q[4][q][a] * r[q] ** 2 * w[q] * Bp[q][p]
                # -int r^2 P~ rho0 [div u + kappa U/r] : row test pressure p, column velocity a
                K[nloc_v + p][a] -= (
                    Bp[q][p] * r[q] ** 2 * rho0[q] * w[q] * Qanel[q][a]
                )

    idx = np.zeros(nloc, dtype=int)
    for j in range(nq):
        idx[j] = oU + e * n + j
        idx[nq + j] = oV + e * n + j
    for p in range(npe):
        idx[nloc_v + p] = oP + e * npe + p
    # Now scatter to the global matrix.
    for a in range(nloc):
        for bb in range(nloc):
            A[idx[a], idx[bb]] += K[a][bb]

    for j in range(nq):
        bvec[oU + e * n + j] += r[j] ** 2 * f[j] * w[j]

We solve the reduced system which excludes the constrained parts (ie $U(c) = U(b) = 0$).

In [11]:
dofs_d = [oU + 0, oU + Nv - 1]
free = []
for i in range(N):
    if i != dofs_d[0] and i != dofs_d[1]:
        free.append(i)
nf = len(free)

In [12]:
Aff = np.zeros((nf, nf))
for ia in range(nf):
    for ja in range(nf):
        Aff[ia, ja] = A[free[ia], free[ja]]
rhs = np.zeros(nf)
for ia in range(nf):
    rhs[ia] = bvec[free[ia]]
xf = np.linalg.solve(Aff, rhs)
x = np.zeros(N)
for ia in range(nf):
    x[free[ia]] = xf[ia]

In [13]:
res = [0.0, 0.0]
for s in range(2):
    for j in range(N):
        res[s] += A[dofs_d[s], j] * x[j]
    res[s] -= bvec[dofs_d[s]]
T_c = -res[0] / c**2
T_b = res[1] / b**2
print(f"T_b = {T_b:.8e}   T_c = {T_c:.8e}")

T_b = 4.06638280e-11   T_c = -1.25174370e-10
